<p><font size="6" color='grey'> <b>
KI-Agenten. Planen. Handeln. Prüfen.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
Capstone
</b></font> </br></p>

---

**Beitrag zum Leitprojekt:** Der Capstone ist kein freies KI-Projekt, sondern der Nachweis, dass ein eigenes System als **kontrolliertes Agentensystem** funktioniert — mit demselben Dreiklang wie der Kurs: **Planen** (Tool-/Worker-Wahl und State), **Handeln** (Tools, RAG und Workflow) und **Prüfen** (Gate/HITL, Evaluation mit Negativfällen). Die Pflichtbestandteile unten sind bewusst an den Capstone-Kriterien der Kurs-Leitaufgabe ausgerichtet. Für Capstone-Varianten kann der `meeting-briefing`-Skill als Referenz für Ausgabeformat, Quellenpflicht und Action-Item-Extraktion dienen.


In [1]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul
!uv pip install --system -q fastapi uvicorn httpx

import os
os.environ["LANGSMITH_TRACING"]  = "true"
os.environ["LANGSMITH_PROJECT"]  = "M38-Capstone"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace
)

setup_api_keys(['OPENAI_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import BASELINE, ROUTER, JUDGE, PLANNER, WORKER, WORKER_PREMIUM, CODING, EMBEDDINGS
# LangSmith Tracing
run_cfg = {
    "run_name": "M38_Capstone",
    "tags": ["m38", "capstone"],
    "metadata": {"notebook": "M38", "version": "1.0"}
}


✓ OPENAI_API_KEY erfolgreich gesetzt

Python Version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

Installierte LangChain- und LangGraph-Bibliotheken:
langchain                                1.2.15
langchain-chroma                         1.1.0
langchain-classic                        1.0.4
langchain-community                      0.4.1
langchain-core                           1.3.1
langchain-ollama                         1.1.0
langchain-openai                         1.2.0
langchain-text-splitters                 1.1.2
langgraph                                1.1.9
langgraph-checkpoint                     4.0.2
langgraph-prebuilt                       1.0.10
langgraph-sdk                            0.3.13

IP-Adresse: 34.46.3.32
Hostname: 32.3.46.34.bc.googleusercontent.com
Stadt: Council Bluffs
Region: Iowa
Land: US
Koordinaten: 41.2619,-95.8608
Provider: AS396982 Google LLC
Postleitzahl: 51502
Zeitzone: America/Chicago


# 1 | Kompakter Kursrückblick

---





Dieses Diagramm strukturiert das Konzept eines Agenten entlang von drei Ebenen: **Eigenschaften**, **Funktionen** und **Infrastruktur**.

Auf der ersten Ebene werden grundlegende **Merkmale** beschrieben, die einen Agenten auszeichnen, etwa Autonomie, Reaktionsfähigkeit oder Zielorientierung.

Daraus leiten sich auf der zweiten Ebene die zentralen **Fähigkeiten** ab, die ein Agent zur Aufgabenerfüllung benötigt – beispielsweise Planung, Nutzung von Werkzeugen oder der Umgang mit Zustand und Feedback.

Die dritte Ebene umfasst schließlich die **technischen** **Rahmenbedingungen**, die einen stabilen und kontrollierten Betrieb ermöglichen, wie Kontextverwaltung, Zugriff auf externe Systeme oder Sicherheitsmechanismen.

Die Zuordnung der Module (Mx) dient dabei als Referenz auf die zugrunde liegenden Inhalte und erlaubt eine gezielte Vertiefung einzelner Aspekte.



In [2]:
#@markdown   <p><font size="4" color='green'>  Agenten-Architektur: Eigenschaften, Funktionen & Infrastruktur</font> </br></p>

diagram = """
%%{init: {'theme':'forest'}}%%
graph TD
    classDef property fill:#ececff,stroke:#9370db,stroke-width:2px,color:#333
    classDef function fill:#e1f5fe,stroke:#01579b,stroke-width:2px,color:#333
    classDef infra fill:#fff3e0,stroke:#e65100,stroke-width:2px,color:#333
    classDef groupStyle fill:#f9f9f9,stroke:#d3d3d3,stroke-dasharray: 5 5

    subgraph E1["<b>🧠 1. Eigenschaften (Was einen Agenten ausmacht)</b>"]
        direction LR
        A(["<b>Autonomie</b><br/>Trifft eigenständig Entscheidungen<br/><sub>M02, M07</sub>"])
        R(["<b>Reaktionsfähigkeit</b><br/>Reagiert auf Änderungen<br/><sub>M01, M05, M09</sub>"])
        P(["<b>Proaktivität</b><br/>Verfolgt Ziele über Zeit<br/><sub>M19–M21</sub>"])
        I(["<b>Interaktionsfähigkeit</b><br/>Interagiert mit Menschen und Systemen<br/><sub>M03, M17</sub>"])
    end

    subgraph E2["<b>🛠️ 2. Funktionen (Was ein Agent können muss)</b>"]
        direction LR
        PL["<b>Planning</b><br/>Zerlegt Ziele in Schritte<br/><sub>M08, M33</sub>"]
        ME["<b>Memory</b><br/>Hält Zustand und Verlauf<br/><sub>M16, M18</sub>"]
        TU["<b>Tool Use</b><br/>Nutzt externe Werkzeuge und APIs<br/><sub>M01, M05, M30</sub>"]
        MO["<b>Monitoring</b><br/>Beobachtet Ergebnisse und Feedback<br/><sub>M15, M24</sub>"]
        DE{{"<b>Delegation (optional)</b><br/>Orchestriert Sub-Agenten<br/><sub>M20, M33, M35, M38</sub>"}}
    end

    subgraph E3["<b>🏗️ 3. Infrastruktur (Was den Betrieb stützt)</b>"]
        direction LR
        CM[/"<b>Context Management</b><br/>Filtert und verdichtet Kontext<br/><sub>M11–M14, M22</sub>"/]
        GR[/"<b>Guardrails</b><br/>Prüft und begrenzt Aktionen<br/><sub>M09, M23</sub>"/]
        EA[/"<b>Environment Access</b><br/>Dateien, APIs, Systeme<br/><sub>M29, M30, M37</sub>"/]
        OB[/"<b>Observability</b><br/>Logs, Traces, Debugging<br/><sub>M15, M24, M37, M38</sub>"/]
    end

    A -.-> PL
    A -.-> TU

    P -.-> PL
    P -.-> ME
    P -.-> DE

    R -.-> MO

    I -.-> TU

    PL --> CM
    ME --> CM

    TU --> EA
    TU --> GR

    MO --> OB
    DE --> OB

    class A,R,P,I property
    class PL,ME,TU,MO,DE function
    class CM,GR,EA,OB infra
    class E1,E2,E3 groupStyle
"""

mermaid(diagram, width=1300)

# 2 | Pattern-Landkarte als Referenz
---


Diese Pattern-Landkarte ist nur Referenz. Für das Capstone zählt ab Abschnitt A vor allem der eigene kontrollierte Workflow mit Tools, RAG/Evidence, Gate/HITL und Evaluation.

Dieser Kurs hat sich Schritt für Schritt durch das Agenten-Ökosystem geführt.
Viele Konzepte haben **Namen** — Pattern-Bezeichnungen aus der AI-Engineering-Literatur.

Das folgende Diagramm zeigt, welche Kursinhalte welchen etablierten Patterns entsprechen.

<p><font color='darkblue' size="4">📝 <b>Hinweis</b></font></p>

Pattern-Namen sind Vokabular — für Gespräche, Interviews, Architektur-Diskussionen. Das Verständnis entsteht durch die praktische Arbeit in den Modulen — die Namen kommen danach.

## Begriffsstandard für das Capstone

Diese Begriffe werden im Capstone konsistent verwendet. Wenn ein Begriff im Projekt anders genutzt wird, muss die Abweichung kurz begründet werden.

| Begriff | Bedeutung im Kurs | Nicht verwechseln mit |
|---|---|---|
| **Chain** | Linearer LCEL-Ablauf ohne eigene Routing-Logik | Graph, Agent |
| **Workflow** | Kontrollierter Ablauf mit mehreren Schritten, oft deterministisch | autonomer Agent |
| **Graph** | LangGraph-State-Machine mit Nodes, Edges, State und Routing | bloße Architektur-Skizze |
| **Agent** | LLM-gesteuerte Einheit, die Entscheidungen über Tools oder nächste Schritte trifft | einfache Chain |
| **Tool** | Aufrufbare Funktion mit Schema, Type Hints und klarer Beschreibung | Skill, Prompt |
| **RAG/Evidence Tool** | Tool, das Dokumente sucht und Quellenstatus liefert | freies Modellwissen |
| **Skill** | Paket aus Instruktionen, Referenzen und optionalen Skripten | einzelnes Tool |
| **Supervisor** | Agent/Node, der Aufgaben an Worker routet und Abschlussbedingungen prüft | Manager-Metapher ohne Codepfad |
| **Planner** | Komponente, die sichtbare Schritte und Stop-Kriterien erzeugt | versteckte Gedankenkette |
| **Gate/HITL** | Prüfschritt oder menschliche Freigabe vor kritischer Ausgabe | bloße Empfehlung im Prompt |

**Reasoning-Regel:** Im Capstone werden keine versteckten Chain-of-Thoughts verlangt. Bewertet werden sichtbare Artefakte: Plan, Tool-Wahl, Evidence-Status, Gate-Entscheidung und Eval-Ergebnis.


In [ ]:
#@markdown   <p><font size="4" color="green">Pattern-Landkarte nach Gruppen</font> </br></p>

diagram = '''
%%{init: {'theme':'forest'}}%%
mindmap
  root((Generative AI
Design Patterns))
    Agent-Grundlagen
      Tool Calling
      Code Execution
      Structured Output
      Dependency Injection
      Prompt Optimization
    Reasoning
      Sichtbare Reasoning-Artefakte
    Output Control
      Style Transfer
      Template Generation
    Memory und Kontrolle
      Persistentes Memory
      Human-in-the-Loop
    RAG und Wissen
      Basic RAG
      Semantic Indexing
      Indexing at Scale
      Index-aware Retrieval
      Node Postprocessing
      Trustworthy Generation
      Deep Search
    Zuverlaessigkeit
      Reflection
      LLM-as-Judge
    Multi-Agent
      Multi-Agent Collaboration
    Deployment
      Small Language Model
      Prompt Caching
      Inference Optimization
      Degradation Testing
    Safety
      Self-Check
      Guardrails
'''

mermaid(diagram, width=1000)


## Pattern-Übersicht

> Quelle: Lakshmanan & Hapke — *Generative AI Design Patterns* (O'Reilly)
> **—** = im Kurs nicht behandelt &nbsp;|&nbsp; **\*** = kursinternes Pattern (nicht im Buch)


### Agent-Grundlagen

| Pattern | Kurz-Erläuterung | Modul |
|---|---|---|
| **Tool Calling** | LLM ruft externe Funktionen und APIs auf | M01, M02, M05 |
| **Code Execution** | Agent führt Code aus und interpretiert Ergebnisse | — |
| **Structured Output** | Typsichere Ausgaben via Pydantic / JSON-Schema | M04 |
| **Dependency Injection** | Kontext und Daten dynamisch in Prompts einbetten | M03 |
| **Prompt Optimization** | Systematische Verbesserung von System-Prompts | M03 |



### Reasoning

| Pattern | Kurz-Erläuterung | Modul |
|---|---|---|
| **Sichtbare Reasoning-Artefakte** | Plan, Tool-Wahl, Evidence-Status und Gate-Entscheidung statt versteckter Gedankenkette | M03, M09 |



### Output Control

| Pattern | Kurz-Erläuterung | Modul |
|---|---|---|
| **Style Transfer** | Textstil via Prompt gezielt verändern | M03 |

| **Template Generation** | Strukturierte Ausgabe-Templates erzwingen | M03, M04 |



### Memory & Kontrolle

| Pattern | Kurz-Erläuterung | Modul |
|---|---|---|
| **Persistentes Memory** | Persistentes Gedächtnis über Sitzungen hinweg | M16, M18 |
| **Human-in-the-Loop** \* | Mensch genehmigt oder korrigiert Agenten-Aktionen | M17 |


### RAG & Wissen

| Pattern | Kurz-Erläuterung | Modul |
|---|---|---|
| **Basic RAG** | Dokumenten-Retrieval + LLM-Antwortgenerierung | M11, M12, M13 |
| **Semantic Indexing** | Vektorbasierte Indexierung für semantische Suche | M11, M12 |
| **Indexing at Scale** | Skalierbare Indizierung großer Dokumentenmengen | M12 |
| **Index-aware Retrieval** | Retrieval mit Kenntnis der Index-Struktur | M13, M14 |
| **Node Postprocessing** | Filtern und Reranken abgerufener Dokumente | M28 |
| **Trustworthy Generation** | Belegbare Antworten mit Quellen-Grounding | M28 |
| **Deep Search** | Mehrschrittige, adaptive Suche mit Agenten | M14, M22, M28 |


### Zuverlässigkeit & Evaluation

| Pattern | Kurz-Erläuterung | Modul |
|---|---|---|
| **Reflection** | Agent prüft und korrigiert eigene Ausgaben | M17, M28 |
| **LLM-as-Judge** | LLM bewertet automatisiert Qualität von Ausgaben | M15, M24 |


### Multi-Agent

| Pattern | Kurz-Erläuterung | Modul |
|---|---|---|
| **Multi-Agent Collaboration** | Spezialisierte Agenten arbeiten koordiniert | M19, M20, M21, M35 |


### Deployment

| Pattern | Kurz-Erläuterung | Modul |
|---|---|---|
| **Small Language Model** | Kleinere Modelle für spezifische Teilaufgaben | M36 |
| **Prompt Caching** | Wiederverwendung gecachter Prompt-Präfixe | M36 |
| **Inference Optimization** | Batching, Quantisierung, Latenzoptimierung | M36 |
| **Degradation Testing** | Belastungs- und Regressionstests im Deployment | M36 |


### Safety

| Pattern | Kurz-Erläuterung | Modul |
|---|---|---|
| **Self-Check** | Agent validiert eigene Ausgaben vor der Rückgabe | M23 |
| **Guardrails** | Sicherheitsschranken für Input und Output | M23 |


# A | Aufgaben

---


<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Das Capstone ist die Abschlussarbeit des Kurses. Es gibt kein vorgegebenes Lösungsmodell — nur Mindestanforderungen und optionale Erweiterungen. Ab hier beginnt der eigentliche Capstone-Arbeitsauftrag; die Rückblicke davor dienen nur als Orientierung.

**Hinweis zur Lösungshilfe:**
> Gemini in Google Colab, LangSmith-Tracing und die Notebooks aus dem Kurs stehen als Unterstützung zur Verfügung.

---

## Pflichtbestandteile

Ausgerichtet an den Capstone-Kriterien der Kurs-Leitaufgabe: Das Projekt wird nicht als allgemeine KI-Anwendung bewertet, sondern als kontrolliertes Agentensystem.

| # | Anforderung | Kriterium |
|---|-------------|-----------|
| 1 | **Architektur-Skizze** (Mermaid) | Alle Komponenten beschriftet, Pattern-Namen angegeben |
| 2 | **Kontrollierter Workflow** | Supervisor + ≥ 2 Worker als StateGraph, insgesamt ≥ 3 Tools (davon mind. 1 RAG/Evidence Tool) |
| 3 | **Strukturierte Ausgabe** | Finale Antwort als Pydantic-Schema (z. B. Antwort, Quelle, Sicherheit) |
| 4 | **Freigabe-/Kontrollpunkt** | Mindestens ein Gate oder HITL-Schritt vor einer kritischen Ausgabe |
| 5 | **Evaluation mit Negativfällen** | ≥ 1 LLM-as-Judge-Kriterium **und** mindestens ein Testfall, bei dem das System korrekt ablehnt oder eskaliert |
| 6 | **Sichtbare Quellen/Traces** | Tool-Aufrufe und Quellen im LangSmith-Trace oder in der Ausgabe nachvollziehbar |

## Optionale Erweiterungen

- Mehrere RAG-Quellen oder Vektordatenbank-Typen
- Gradio UI für den Agenten (Quellen, Trace, Freigabe sichtbar)
- Fehlertoleranz (Retry-Logik, Fallback-Agenten)
- LangSmith Evaluation Dataset mit Baseline-Score

## Bewertungskriterien

| Kriterium | Punkte |
|-----------|--------|
| Architektur sinnvoll & begründet | 2 |
| Pattern-Namen korrekt verwendet | 1 |
| Code lauffähig & strukturiert | 2 |
| Strukturierte Ausgabe vorhanden | 1 |
| Gate/HITL-Schritt vorhanden | 2 |
| Evaluation inkl. Negativfall implementiert | 2 |
| **Gesamt** | **10** |

## Abgabeformat

- Lauffähiges Jupyter Notebook (alle Zellen ausgeführt)
- Mermaid-Diagramm als erste Zelle im Implementierungsblock
- Kurze Markdown-Zelle am Ende mit zwei Reflexionsfragen: Was würdest du als nächstes verbessern? Wann darf der Agent nicht autonom handeln?

**✅ Erledigt wenn:** Das Notebook läuft von oben nach unten fehlerfrei durch; Mermaid-Diagramm, Pattern-Tabelle, strukturierte Ausgabe, Gate/HITL-Schritt und mindestens ein LLM-as-Judge-Ergebnis mit Negativfall sind vorhanden.


## Verweis-Matrix für die Umsetzung

| Capstone-Baustein | Wiederverwenden aus | Mindestnachweis im Capstone |
|---|---|---|
| Tool-Vertrag und robuste Tools | M01, M05, M10 | `@tool`, Type Hints, isolierter Tool-Test |
| RAG/Evidence Tool | M14, M22, M28 | Antwort nennt Quelle oder lehnt als `Nicht im Korpus.` ab |
| Kontrollierter Workflow | M08-M10, M19-M21 | `StateGraph` mit Supervisor, mindestens zwei Worker-Pfaden und klarer Endbedingung |
| Strukturierte Ausgabe | M04 | Pydantic-Schema für finale Antwort |
| Gate/HITL | M09, M17, M23 | Gate vor kritischer Ausgabe oder echter `interrupt()`-Schritt |
| Memory / Sessions | M16, M18 | `thread_id`, Checkpointer oder bewusst begründeter Verzicht auf Memory |
| Evaluation mit Negativfall | M15, M24 | Mindestens ein Judge-/Eval-Kriterium und ein Out-of-Corpus-Test |
| Kosten und Betrieb | M25, M36, M37 | Tool-/Token-Budget, Trace oder Monitoring-Hinweis |
| DeepAgents-Variante | M31-M35 | Nur verwenden, wenn Planning, Filesystem oder Skills wirklich gebraucht werden |

Diese Matrix ist die Checkliste für die Implementierungssektion: Jeder Pflichtbaustein soll auf ein früheres Kursmuster zurückführbar und im Notebook sichtbar testbar sein.


## Starter-Setup: Korpus und Eval-Set

Nutzen Sie diesen Block als Ausgangspunkt für den Capstone des Meeting- & Research-Briefing-Agenten.

**Hinweis:** Der spätere Block `Capstone-Implementierung` ist absichtlich ein Starter-Gerüst für Teilnehmende. Er ist kein unvollständiges Referenzsystem; die ausgearbeitete Meeting-Agent-Demo steht davor als Beispiel, die Capstone-Lösung soll eigenständig entstehen.


In [ ]:
from pathlib import Path
import json
import shutil
from genai_lib.utilities import copy_from_github

KORPUS_QUELLE = "ralf-42/Agenten/02_daten/01_text/korpus_meeting_briefing"
KORPUS_MASKE = "*.pdf"
KORPUS_TARGET = "/content/files"

EVAL_QUELLE = "ralf-42/Agenten/02_daten/05_sonstiges"
EVAL_MASKE = "eval_meeting_briefing*.json"
EVAL_TARGET = "/content/eval"

shutil.rmtree(KORPUS_TARGET, ignore_errors=True)
copy_from_github(
    source=KORPUS_QUELLE,
    target=KORPUS_TARGET,
    mask=KORPUS_MASKE,
)

shutil.rmtree(EVAL_TARGET, ignore_errors=True)
copy_from_github(
    source=EVAL_QUELLE,
    target=EVAL_TARGET,
    mask=EVAL_MASKE,
)

with open(Path(EVAL_TARGET) / "eval_meeting_briefing.json", encoding="utf-8") as f:
    eval_meeting_briefing = json.load(f)

print(f"Korpus geladen nach: {KORPUS_TARGET}")
print(f"Eval-Fragen geladen: {len(eval_meeting_briefing)}")


In [ ]:
# Capstone-Implementierung
# Tipp: Mermaid-Diagramm als Blaupause verwenden

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

# 1. Modelle initialisieren
# supervisor_llm = init_chat_model(JUDGE)   # gpt-5.4 empfohlen
# worker_llm     = init_chat_model(WORKER)  # gpt-5.4-mini, kein temperature
# judge_llm      = init_chat_model(JUDGE)   # gpt-5.4 empfohlen

# 2. Tools definieren
# at.tool
# def mein_tool(eingabe: str) -> str: ...

# 3. Agenten aufbauen (Supervisor + mindestens 2 Worker)

# 4. StateGraph zusammensetzen

# 5. LangSmith-Tracing konfigurieren
# import os
# os.environ['LANGSMITH_TRACING'] = 'true'
# os.environ['LANGSMITH_PROJECT'] = 'capstone'

# 6. LLM-as-Judge für kritische Stelle

# 7. System testen
# result = mein_system.invoke({'input': 'Testaufgabe'}, config=run_cfg)
# print(result)

# --- Was würde ich als nächstes verbessern? ---
# ausblick = '...'

## Minimaler End-to-End-Smoke-Test

Der folgende Smoke-Test ist keine fertige Capstone-Lösung. Er prüft nur, ob die Mindestlogik technisch zusammenpasst: Evidence-Tool, strukturierte Ausgabe, Gate-Entscheidung und Negativfall-Evaluation. Die eigentliche Capstone-Lösung muss danach den vollständigen Supervisor-Workflow mit mindestens zwei Workern ausbauen.


In [ ]:
# Minimaler End-to-End-Smoke-Test fuer die Capstone-Pflichtbestandteile
from typing import Literal, TypedDict
from pydantic import BaseModel, Field
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END


class BriefingAntwort(BaseModel):
    antwort: str = Field(description="Kurze fachliche Antwort oder klare Ablehnung")
    quelle: str = Field(description="Quelldokument oder 'Nicht im Korpus'")
    sicherheit: Literal["hoch", "mittel", "niedrig"]
    human_review_noetig: bool


class SmokeState(TypedDict, total=False):
    frage: str
    route: str
    evidence: str
    final: BriefingAntwort
    eval_ok: bool


@tool
def evidence_search(frage: str) -> str:
    """Sucht im Mini-Korpus des Smoke-Tests nach belastbarer Evidenz."""
    frage_lower = frage.lower()
    if "r2" in frage_lower:
        return "Quelle: protokoll_steuerkreis_2026-04-28.pdf | Risiko R2 ist mitigiert."
    if "umsatzzahlen" in frage_lower:
        return "Nicht im Korpus."
    return "Quelle: protokoll_steuerkreis_2026-03-17.pdf | Entscheidung D1 evaluiert ChromaDB fuer den Prototyp."


# Isolierter Tool-Test: Tools zuerst pruefen, bevor sie im Graphen stecken.
assert "Quelle:" in evidence_search.invoke({"frage": "Was ist Entscheidung D1?"})
assert evidence_search.invoke({"frage": "Welche Umsatzzahlen gibt es?"}) == "Nicht im Korpus."


def supervisor_node(state: SmokeState) -> dict:
    frage = state["frage"].lower()
    route = "research_worker" if any(signal in frage for signal in ["r2", "entscheidung", "chroma", "umsatz"]) else "general_worker"
    return {"route": route}


def route_from_supervisor(state: SmokeState) -> str:
    return state["route"]


def research_worker(state: SmokeState) -> dict:
    return {"evidence": evidence_search.invoke({"frage": state["frage"]})}


def general_worker(state: SmokeState) -> dict:
    return {"evidence": "Nicht im Korpus."}


def gate_node(state: SmokeState) -> dict:
    evidence = state["evidence"]
    if evidence == "Nicht im Korpus.":
        final = BriefingAntwort(
            antwort="Nicht im Korpus.",
            quelle="Nicht im Korpus",
            sicherheit="niedrig",
            human_review_noetig=True,
        )
    else:
        quelle, aussage = evidence.split(" | ", maxsplit=1)
        final = BriefingAntwort(
            antwort=aussage,
            quelle=quelle.replace("Quelle: ", ""),
            sicherheit="hoch",
            human_review_noetig=False,
        )
    return {"final": final}


def eval_node(state: SmokeState) -> dict:
    final = state["final"]
    frage_lower = state["frage"].lower()
    erwartet_ooc = "umsatzzahlen" in frage_lower
    eval_ok = (final.antwort == "Nicht im Korpus.") if erwartet_ooc else final.quelle.endswith(".pdf")
    return {"eval_ok": eval_ok}


builder = StateGraph(SmokeState)
builder.add_node("supervisor", supervisor_node)
builder.add_node("research_worker", research_worker)
builder.add_node("general_worker", general_worker)
builder.add_node("gate", gate_node)
builder.add_node("eval", eval_node)
builder.add_edge(START, "supervisor")
builder.add_conditional_edges(
    "supervisor",
    route_from_supervisor,
    {"research_worker": "research_worker", "general_worker": "general_worker"},
)
builder.add_edge("research_worker", "gate")
builder.add_edge("general_worker", "gate")
builder.add_edge("gate", "eval")
builder.add_edge("eval", END)
capstone_smoke_graph = builder.compile()

positive_smoke = capstone_smoke_graph.invoke({"frage": "Welchen Status hat Risiko R2?"})
negative_smoke = capstone_smoke_graph.invoke({"frage": "Was sagt der Korpus zu den Umsatzzahlen?"})

print(positive_smoke["final"].model_dump())
print(negative_smoke["final"].model_dump())
assert positive_smoke["eval_ok"] is True
assert negative_smoke["eval_ok"] is True
assert negative_smoke["final"].human_review_noetig is True


**Praxis-Transfer: Meeting- & Research-Briefing-Agent**

Abschlussprojekt anhand des Meeting- & Research-Briefing-Agent.

1. Welche reale Arbeitsfrage löst Ihr Capstone im Meeting- & Research-Briefing-Agent?
2. Welche Daten, Dokumente oder Tools braucht der Baustein?
3. Welche Ausgabe sollte das System liefern, damit Quellen, Unsicherheit und Entscheidungen nachvollziehbar bleiben?
4. Welche Risiken oder Grenzen bleiben trotz Agentenworkflow?
5. Wann wäre Human Review nötig?
6. Ist der passende Lösungsweg hier Prompt, strukturierte Ausgabe, RAG, Tool, Agent oder Workflow?

<p><font color='darkblue' size="4">
 <b>Viz</b>
</font></p>

- [KI-Agenten-Architektur](https://editor.p5js.org/ralf.bendig.rb/full/Viso2emNI)
- [RAG-Pipeline](https://editor.p5js.org/ralf.bendig.rb/full/RrfB3nCwK)
- [LangGraph](https://editor.p5js.org/ralf.bendig.rb/full/EUzaFq4C4)
- [Checkliste Automatisierung](https://editor.p5js.org/ralf.bendig.rb/full/ckiLlKrql)
- [KI-Prozessoptimierung](https://editor.p5js.org/ralf.bendig.rb/full/xGKXCTR7T)
- [DSPy](https://editor.p5js.org/ralf.bendig.rb/full/Q_dUC2-lV)



# B | Dokumente zum Weiterlesen
---

Ergänzende Artikel aus der Kurs-Dokumentation:

- [Meeting- & Research-Briefing-Agent im Betrieb](https://ralf-42.github.io/Agenten/08-deployment-betrieb/meeting-research-briefing-agent.html)
- [Checkliste Agentensystem](https://ralf-42.github.io/Agenten/04-agenten-implementierung/checkliste-agentensystem.html)
- [Aus Entwicklung ins Deployment](https://ralf-42.github.io/Agenten/08-deployment-betrieb/aus-entwicklung-ins-deployment.html)
- [Evaluation & Observability](https://ralf-42.github.io/Agenten/07-qualitaet-sicherheit/evaluation-observability.html)
- [Agent Security](https://ralf-42.github.io/Agenten/07-qualitaet-sicherheit/agent-security.html)
- [Minimum Viable Agent Stack](https://ralf-42.github.io/Agenten/08-deployment-betrieb/minimum-viable-agent-stack.html)
- [Code Standards](https://ralf-42.github.io/Agenten/10-ressourcen/standards.html)
- [Interaktive Visualisierungen](https://ralf-42.github.io/Agenten/10-ressourcen/interaktive-visualisierungen.html)
- [Links](https://ralf-42.github.io/Agenten/10-ressourcen/links.html)
- [EU AI Act](https://ralf-42.github.io/Agenten/09-regulatorik-verantwortung/eu-ai-act.html)
- [KI-Agenten in regulierten Branchen](https://ralf-42.github.io/Agenten/09-regulatorik-verantwortung/ki-agenten-in-regulierten-branchen.html)
- [Lernpfad](https://ralf-42.github.io/Agenten/lernpfad.html)
- [Start](https://ralf-42.github.io/Agenten/index.html)

